# SmartCare Hospital AI Coursework
## Task 05 — Model Development and Optimization

**Objective:** Compare four classification pipelines using the compact Task 03 feature set. Scaling, feature selection, hyperparameter review, model selection, and threshold selection use training data only. The test set is reserved for Task 06.

## 5.1 Import Libraries

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    cross_val_predict, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

## 5.2 Load Task 03 Outputs

In [2]:
df_model = pd.read_csv('smartcare_preprocessed_unscaled.csv')
fairness_cols = pd.read_csv('fairness_columns.csv')

assert df_model.shape == (875, 25)
assert df_model.isnull().sum().sum() == 0
assert len(df_model) == len(fairness_cols)
assert 'long_wait_flag' not in df_model.columns
assert {'gender', 'age_group'}.issubset(fairness_cols.columns)

print('Model dataset:', df_model.shape)
print('Model inputs:', df_model.shape[1] - 1)
print('Fairness attributes are separate from model inputs.')

Model dataset: (875, 25)
Model inputs: 24
Fairness attributes are separate from model inputs.


## 5.3 Create Independent Data Splits

In [3]:
X = df_model.drop(columns='no_show')
y = df_model['no_show'].astype(int)

X_temp, X_test, y_temp, y_test, fair_temp, fair_test = train_test_split(
    X, y, fairness_cols,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val, fair_train, fair_val = train_test_split(
    X_temp, y_temp, fair_temp,
    test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

cv_strategy = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)

print('Train:', len(X_train), 'Validation:', len(X_val), 'Test:', len(X_test))
print('Train balance:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Test set evaluated in Task 05: No')

Train: 525 Validation: 175 Test: 175
Train balance: {1: 0.566, 0: 0.434}
Test set evaluated in Task 05: No


**Justification:** The 60/20/20 stratified split provides a training set for cross-validation, a validation set for later calibration and supporting checks, and an untouched test set for one final evaluation.

## 5.4 Review the Number of Selected Features

In [4]:
k_rows = []
for k_value in [15, 20, 'all']:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=k_value)),
        ('model', RandomForestClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=1
        ))
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv_strategy,
        scoring={'Accuracy': 'accuracy', 'F1 Score': 'f1', 'ROC-AUC': 'roc_auc'},
        n_jobs=-1
    )
    k_rows.append({
        'k': k_value,
        'CV Mean Accuracy': scores['test_Accuracy'].mean(),
        'CV Mean F1 Score': scores['test_F1 Score'].mean(),
        'CV Mean ROC-AUC': scores['test_ROC-AUC'].mean()
    })

k_comparison_df = (pd.DataFrame(k_rows)
                   .sort_values(['CV Mean F1 Score', 'CV Mean Accuracy'], ascending=False)
                   .reset_index(drop=True))
display(k_comparison_df.round(4))
best_k = k_comparison_df.loc[0, 'k']
print('Selected k:', best_k)

,k,CV Mean Accuracy,CV Mean F1 Score,CV Mean ROC-AUC
0,all,0.5790,0.6610,0.5816
1,20,0.5771,0.6492,0.5761
2,15,0.5657,0.6337,0.5562


Selected k: all


**Justification:** Feature selection is fitted inside every training fold. Only three clear options are compared to avoid unnecessary searching and overfitting.

## 5.5 Random Forest Hyperparameter Review

In [5]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=best_k)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
])

parameter_options = {
    'model__n_estimators': [200, 300, 500],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2'],
    'model__class_weight': ['balanced', 'balanced_subsample']
}

rf_search = RandomizedSearchCV(
    rf_pipeline, parameter_options, n_iter=10,
    scoring='f1', refit=True, cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1,
    return_train_score=False
)
rf_search.fit(X_train, y_train)

baseline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=best_k)),
    ('model', RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=1
    ))
])
baseline_score = cross_validate(
    baseline_rf, X_train, y_train,
    cv=cv_strategy, scoring='f1', n_jobs=-1
)['test_score'].mean()

if rf_search.best_score_ > baseline_score:
    final_rf = rf_search.best_estimator_
    rf_decision = 'Randomized-search configuration retained'
else:
    final_rf = baseline_rf
    rf_decision = 'Baseline balanced configuration retained'

print('Baseline RF CV F1:', round(baseline_score, 4))
print('Best search CV F1:', round(rf_search.best_score_, 4))
print('Decision:', rf_decision)
print('Best searched parameters:', rf_search.best_params_)

Baseline RF CV F1: 0.661
Best search CV F1: 0.661
Decision: Baseline balanced configuration retained
Best searched parameters: {'model__n_estimators': 300, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None, 'model__class_weight': 'balanced'}


## 5.6 Compare Four Models with Training-Only Cross-Validation

In [6]:
def make_pipeline(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=best_k)),
        ('model', model)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(
            max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE
        )
    ),
    'Random Forest': final_rf,
    'Gradient Boosting': make_pipeline(
        GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)
    ),
    'SVM (RBF)': make_pipeline(
        SVC(kernel='rbf', class_weight='balanced', probability=True,
            random_state=RANDOM_STATE)
    )
}

scoring = {
    'Accuracy': 'accuracy', 'Precision': 'precision',
    'Recall': 'recall', 'F1 Score': 'f1', 'ROC-AUC': 'roc_auc'
}
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
cv_rows = []

for name, pipeline in models.items():
    fit_params = {'model__sample_weight': sample_weights} if name == 'Gradient Boosting' else None
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv_strategy,
        scoring=scoring, n_jobs=-1, params=fit_params
    )
    cv_rows.append({
        'Model': name,
        **{f'CV Mean {metric}': scores[f'test_{metric}'].mean() for metric in scoring},
        'CV F1 Std': scores['test_F1 Score'].std()
    })

cv_df = (pd.DataFrame(cv_rows)
         .sort_values(['CV Mean F1 Score', 'CV Mean Accuracy'], ascending=False)
         .reset_index(drop=True))
display(cv_df.round(4))

selected_model_name = cv_df.loc[0, 'Model']
print('Selected model by training-only CV F1:', selected_model_name)

,Model,CV Mean Accuracy,CV Mean Precision,CV Mean Recall,CV Mean F1 Score,CV Mean ROC-AUC,CV F1 Std
0,Random Forest,0.5790,0.6064,0.7272,0.6610,0.5816,0.0373
1,Logistic Regression,0.5905,0.6529,0.6028,0.6245,0.6314,0.0174
2,Gradient Boosting,0.5486,0.5936,0.6327,0.6113,0.5358,0.0446
3,SVM (RBF),0.5676,0.6295,0.5823,0.6041,0.5929,0.0378


Selected model by training-only CV F1: Random Forest


**Justification:** Mean F1-score is primary because the system should detect likely no-shows without creating too many false alerts. Accuracy, precision, recall, and ROC-AUC are supporting measures.

## 5.7 Select a Decision Threshold from Training Data

In [7]:
selected_pipeline = models[selected_model_name]
selected_fit_params = {'model__sample_weight': sample_weights} if selected_model_name == 'Gradient Boosting' else None

oof_probability = cross_val_predict(
    selected_pipeline, X_train, y_train,
    cv=cv_strategy, method='predict_proba',
    n_jobs=-1, params=selected_fit_params
)[:, 1]

threshold_rows = []
for threshold in np.arange(0.35, 0.66, 0.01):
    prediction = (oof_probability >= threshold).astype(int)
    threshold_rows.append({
        'Threshold': threshold,
        'OOF Accuracy': accuracy_score(y_train, prediction),
        'OOF Precision': precision_score(y_train, prediction),
        'OOF Recall': recall_score(y_train, prediction),
        'OOF F1 Score': f1_score(y_train, prediction)
    })

threshold_df = pd.DataFrame(threshold_rows)
default_row = threshold_df.iloc[(threshold_df['Threshold'] - 0.50).abs().argmin()]
eligible = threshold_df[
    (threshold_df['OOF Accuracy'] >= default_row['OOF Accuracy']) &
    (threshold_df['OOF Recall'] >= 0.60)
]
best_row = eligible.sort_values(['OOF F1 Score', 'OOF Accuracy'], ascending=False).iloc[0]
decision_threshold = round(float(best_row['Threshold']), 2)

display(threshold_df.sort_values(['OOF F1 Score', 'OOF Accuracy'], ascending=False).head(8).round(4))
print('Selected threshold:', decision_threshold)

,Threshold,OOF Accuracy,OOF Precision,OOF Recall,OOF F1 Score
1,0.36,0.5771,0.5793,0.9226,0.7117
0,0.35,0.5733,0.5765,0.9259,0.7106
2,0.37,0.5733,0.5778,0.9125,0.7076
3,0.38,0.5752,0.5801,0.9024,0.7062
4,0.39,0.5790,0.5837,0.8923,0.7057
5,0.40,0.5752,0.5822,0.8822,0.7015
7,0.42,0.5695,0.5805,0.8620,0.6938
6,0.41,0.5638,0.5766,0.8620,0.6910


Selected threshold: 0.39


## 5.8 Fit Models and Check Validation Performance

In [10]:
fitted_models = {}
validation_rows = []

for name, pipeline in models.items():
    if name == 'Gradient Boosting':
        pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)
    else:
        pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

    probability = pipeline.predict_proba(X_val)[:, 1]
    threshold = decision_threshold if name == selected_model_name else 0.50
    prediction = (probability >= threshold).astype(int)
    validation_rows.append({
        'Model': name, 'Decision Threshold': threshold,
        'Accuracy': accuracy_score(y_val, prediction),
        'Precision': precision_score(y_val, prediction),
        'Recall': recall_score(y_val, prediction),
        'F1 Score': f1_score(y_val, prediction),
        'ROC-AUC': roc_auc_score(y_val, probability)
    })

validation_df = pd.DataFrame(validation_rows).sort_values('F1 Score', ascending=False)
display(validation_df.round(4))
print('Selected model remains:', selected_model_name)

,Model,Decision Threshold,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,Random Forest,0.39,0.6057,0.5949,0.9495,0.7315,0.6033
3,SVM (RBF),0.50,0.5829,0.5956,0.8182,0.6894,0.5941
2,Gradient Boosting,0.50,0.6057,0.6389,0.6970,0.6667,0.6166
0,Logistic Regression,0.50,0.6057,0.6630,0.6162,0.6387,0.6640


Selected model remains: Random Forest


## 5.9 Save Models and Required Data

In [9]:
for name, pipeline in fitted_models.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(pipeline, MODEL_DIR / f'{filename}_pipeline.pkl')

joblib.dump(fitted_models[selected_model_name], MODEL_DIR / 'selected_model_pipeline.pkl')

X_train.to_csv(MODEL_DIR / 'X_train.csv', index=False)
X_val.to_csv(MODEL_DIR / 'X_val.csv', index=False)
X_test.to_csv(MODEL_DIR / 'X_test.csv', index=False)
y_val.to_csv(MODEL_DIR / 'y_val.csv', index=False)
y_test.to_csv(MODEL_DIR / 'y_test.csv', index=False)
fair_test.reset_index(drop=True).to_csv(MODEL_DIR / 'fair_test.csv', index=False)

k_comparison_df.to_csv(MODEL_DIR / 'feature_count_comparison.csv', index=False)
cv_df.to_csv(MODEL_DIR / 'cross_validation_comparison.csv', index=False)
threshold_df.to_csv(MODEL_DIR / 'threshold_optimization.csv', index=False)
validation_df.to_csv(MODEL_DIR / 'validation_check.csv', index=False)

metadata = {
    'selected_model': selected_model_name,
    'selection_metric': 'Mean F1-score from training-only 5-fold stratified cross-validation',
    'feature_columns': X_train.columns.tolist(),
    'feature_selection_k': best_k,
    'decision_threshold': decision_threshold,
    'random_forest_decision': rf_decision,
    'random_state': RANDOM_STATE,
    'train_size': len(X_train),
    'validation_size': len(X_val),
    'test_size': len(X_test),
    'test_evaluated_in_task05': False
}
with open(MODEL_DIR / 'selected_model_info.json', 'w') as file:
    json.dump(metadata, file, indent=2)

print('Models and required Task 05 files saved successfully.')

Models and required Task 05 files saved successfully.


### Task 05 Summary

- The compact 24-feature dataset was used without artificial resampling.
- Scaling and feature selection were fitted inside training folds.
- Four models were compared using training-only five-fold stratified cross-validation.
- Random Forest tuning was kept only if it improved CV F1.
- The threshold was selected from out-of-fold training probabilities.
- Validation was a supporting check only, and the test set remained untouched.